# 01 — GTZAN Exploratory Data Analysis

**Where this runs:** Kaggle (with the GTZAN dataset attached) or locally once
you've run `scripts/download_gtzan.py`.

## What this notebook does
A quick look at the data before modelling:
1. Load the committed split manifest and check class balance per split.
2. Plot a sample waveform for one clip.
3. Plot its mel-spectrogram (the exact representation the CNN will learn from).

Nothing here trains a model — it's just to build intuition and to sanity-check
that the data and the split look right.

In [ ]:
# On Kaggle the repo isn't on the path; add the project root so `import src` works.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent  # notebooks/ -> repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display

from src.utils import load_config, PROJECT_ROOT
from src.data.manifest import load_clips

cfg = load_config('data')
SR = cfg['dataset']['sample_rate']
# Point this at the attached Kaggle dataset path, or the local download.
AUDIO_DIR = PROJECT_ROOT / cfg['dataset']['audio_dir']
print('audio dir:', AUDIO_DIR)

## 1. Class balance per split
GTZAN is balanced by construction (100 clips/genre), so this mostly confirms
the stratified split kept that balance across train/val/test.

In [ ]:
clips = load_clips()  # all splits
df = pd.DataFrame([{'genre': c.genre, 'split': c.split} for c in clips])
counts = df.pivot_table(index='genre', columns='split', aggfunc=len, fill_value=0)
counts = counts[['train', 'val', 'test']]
display(counts)

counts.plot(kind='bar', stacked=True, figsize=(10, 4))
plt.title('Clips per genre by split')
plt.ylabel('# clips')
plt.tight_layout(); plt.show()

## 2. Sample waveform
The raw audio signal: amplitude over time for one clip.

In [ ]:
sample = next(c for c in clips if c.genre == 'blues')
y, sr = librosa.load(sample.path(AUDIO_DIR), sr=SR, mono=True)

plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title(f'Waveform — {sample.clip_id}')
plt.tight_layout(); plt.show()

## 3. Mel-spectrogram (what the CNN sees)
A mel-spectrogram shows energy across mel-scaled frequency bands over time.
The CNN in Phase 4 treats this 2-D image as its input. Params come straight
from `configs/features.yaml` so this preview matches training exactly.

In [ ]:
fcfg = load_config('features')['melspec']
S = librosa.feature.melspectrogram(
    y=y, sr=sr,
    n_mels=fcfg['n_mels'], n_fft=fcfg['n_fft'], hop_length=fcfg['hop_length'],
)
S_db = librosa.power_to_db(S, ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(S_db, sr=sr, hop_length=fcfg['hop_length'],
                         x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title(f'Mel-spectrogram — {sample.clip_id}  (shape={S_db.shape})')
plt.tight_layout(); plt.show()